# LeetCode #1130 — Minimum Cost Tree From Leaf Values

**Medium · Python 3 · Binary Tree / Interval DP / Greedy / Monotonic Stack**  
[Đề gốc LeetCode](https://leetcode.com/problems/minimum-cost-tree-from-leaf-values/) · [Đối chiếu ví dụ, điều kiện và các hướng giải](https://leetcode.doocs.org/en/lc/1130/)

```python
class Solution:
    def mctFromLeafValues(self, arr: List[int]) -> int:
        ...
```

**Mục tiêu:** hiểu cây nhị phân qua ví dụ 3 lá, học cách chia đoạn và quy hoạch động `O(n³)` trước, sau đó mới tiếp cận quy tắc tham lam + **monotonic stack `O(n)`**. Không cần biết trước Tree DP hay stack.

## 1. Kết quả học tập

1. Phân biệt **leaf (nút lá)** và **internal node (nút trong)**.
2. Hiểu lá phải theo thứ tự trái→phải **đúng như `arr`**, không được tự đảo.
3. Với mỗi nút trong, tính đúng `max(lá trái) × max(lá phải)`.
4. Từ việc thử mọi cách chia, lập được công thức **DP trên đoạn**.
5. Giải thích được phép `push/pop` của **stack đơn điệu giảm** và vì sao phần tử nhỏ được ghép với hàng xóm lớn hơn gần nhất phù hợp.
6. Viết code LeetCode `O(n)` và kiểm chứng bằng DP (oracle).

## 2. Đề bài (diễn giải sát nghĩa) và constraints

Cho mảng **số nguyên dương** `arr`. Xét các **cây nhị phân đầy đủ** (*full binary trees*): mỗi nút có **0 hoặc 2 con**, không có nút đúng 1 con. Các lá đọc **in-order** (trái→gốc→phải, riêng lá chính là các giá trị được gặp từ trái sang phải) phải có dãy giá trị **đúng bằng `arr`**.

Giá trị của **mỗi nút trong** bằng:

`giá trị nút = max(giá trị các lá bên trái) × max(giá trị các lá bên phải)`.

Yêu cầu **trả tổng các giá trị của nút trong nhỏ nhất**, **không phải** trả về đối tượng cây.

- `[6,2,4]` → **32**.
- `[4,11]` → **44**.

**Constraints:** `2 <= arr.length <= 40`; `1 <= arr[i] <= 15`; tổng tối ưu vừa trong số nguyên signed 32-bit.

## 3. Nền tảng: cây 2 con / nút lá / nút trong

Nút **lá** không có con. Nút **trong** có đúng 2 con theo ràng buộc đề.

```text
    (nút trong)            (nút trong)
     /       \             /       \
  [lá 6]  (nút trong)  (nút trong)  [lá 4]
          /      \      /      \
       [lá 2] [lá 4]  [lá 6]   [lá 2]
```

Hai hình có cùng thứ tự lá `[6,2,4]`, nhưng cách ghép nhóm khác nên tổng có thể khác. Không được đảo thành `[2,4,6]` để làm bài dễ hơn.

## 4. Tính tay toàn bộ hai cây cho `[6,2,4]`

**Cây A: `(6,(2,4))`**

- Nút trong dưới: `2×4=8`.
- Nút trong trên: `max([6])×max([2,4]) = 6×4=24`.
- Tổng = `8+24=32`.

**Cây B: `((6,2),4)`**

- Nút trong dưới: `6×2=12`.
- Nút trong trên: `max([6,2])×max([4]) = 6×4=24`.
- Tổng = `12+24=36`.

Chọn **32**, chứ không chọn hình nhìn “cân đối” hơn. Với n lớn, số dạng cây tăng rất nhanh; cần thuật toán thay cho vẽ toàn bộ.

In [1]:
arr_demo = [6,2,4]
cost_a = 2*4 + 6*max(2,4)
cost_b = 6*2 + max(6,2)*4
print('Cây (6,(2,4)):',cost_a)
print('Cây ((6,2),4):',cost_b)
assert (cost_a,cost_b) == (32,36)

Cây (6,(2,4)): 32
Cây ((6,2),4): 36


## 5. Python: slice, `max`, `min`, vòng lặp theo điểm chia

`arr[i:j+1]` lấy các phần tử chỉ số `i`...`j` (kể cả j). Khi chọn điểm chia `k`, ta có hai đoạn **không rỗng**:

- trái: `arr[i:k+1]`;
- phải: `arr[k+1:j+1]`.

Bởi vậy `k` chỉ chạy từ `i` đến `j-1`.

`max(arr[i:k+1])` là **giá trị lớn nhất của lá**, không phải tổng đoạn.

In [2]:
arr_demo = [6,2,4]
for k in range(len(arr_demo)-1):
    left = arr_demo[:k+1]
    right = arr_demo[k+1:]
    print('k =',k,'| trái:',left,'| phải:',right,
          '| tích max lá =',max(left)*max(right))
assert max(arr_demo[1:]) == 4

k = 0 | trái: [6] | phải: [2, 4] | tích max lá = 24
k = 1 | trái: [6, 2] | phải: [4] | tích max lá = 24


## 6. Cách trực giác — thử mọi cách chia (Brute Force)

Chọn một điểm chia k thành **cây con trái** và **cây con phải**, rồi giải độc lập mỗi bên. Nếu hai cây con có tổng tối ưu `cost_left`, `cost_right` thì:

`cost = cost_left + cost_right + max_left × max_right`.

Duyệt hết các k, chọn nhỏ nhất. Cách này diễn đạt đúng cấu trúc cây nhưng **lặp lại bài toán trên cùng một đoạn** rất nhiều lần. Chỉ dùng demo trên dãy ngắn.

In [3]:
def mct_bruteforce(arr):
    def solve(part):
        if len(part)==1:
            return 0,part[0]  # (tổng nút trong, max lá)
        best = float('inf')
        for k in range(1,len(part)):
            left_cost,left_max = solve(part[:k])
            right_cost,right_max = solve(part[k:])
            best = min(best,left_cost+right_cost+left_max*right_max)
        return best,max(part)
    return solve(arr)[0]

for a in ([6,2,4], [4,11], [1,2,3], [3,3,3]):
    print(a,'->',mct_bruteforce(a))
assert mct_bruteforce([6,2,4])==32
assert mct_bruteforce([4,11])==44

[6, 2, 4] -> 32
[4, 11] -> 44
[1, 2, 3] -> 8
[3, 3, 3] -> 18


## 7. Vấn đề của vét cạn: các đoạn lặp lại

Ví dụ với `[6,2,4,5]`, đoạn `[2,4]` có thể xuất hiện khi chia cây gốc ở nhiều điểm k khác nhau. Vét cạn giải lại cùng một câu hỏi: “chi phí tốt nhất của đoạn `[2,4]` là gì?”.

**Dynamic Programming (DP)** là cách ghi nhớ câu trả lời cho những *bài toán con* trùng nhau, tránh tính lại. Ở đây đơn vị bài toán con là một đoạn chỉ số **`[i..j]`**, nên gọi là **Interval DP — Quy hoạch động trên đoạn**.

## 8. Trạng thái DP và điều kiện biên

`dp[i][j]` = **tổng nút trong nhỏ nhất** để tạo cây từ các lá `arr[i] ... arr[j]` theo đúng thứ tự.

`mx[i][j]` = **giá trị lớn nhất trong các lá** `arr[i..j]`.

Với một lá duy nhất (i=j), không có nút trong nên `dp[i][i]=0`. Ta có `mx[i][i]=arr[i]`.

Với đoạn dài ≥2, mọi cây nhị phân đầy đủ đều có một điểm chia k ở gốc. Duyệt mọi k rồi lấy tối thiểu — đây chính là **nguyên lý tối ưu của DP**.

## 9. Công thức chuyển trạng thái — tự suy ra

Với `i < j` và `i <= k < j`:

\[
\operatorname{cost}(i,k,j)
=dp[i][k]+dp[k+1][j]+mx[i][k]\cdot mx[k+1][j].
\]

Lấy giá trị nhỏ nhất trên k:

\[
dp[i][j]=\min_{i\le k<j}\operatorname{cost}(i,k,j),\qquad dp[i][i]=0.
\]

Dùng `mx[i][j]=max(mx[i][j-1],arr[j])` để tính nhanh giá trị lớn nhất mỗi đoạn. Không được dùng giá trị nút trong thay cho **max của lá**.

In [4]:
# Thử công thức bằng tay trên toàn bộ mảng [6,2,4].
arr_demo = [6,2,4]
split_0 = 0 + (2*4) + 6*max(2,4)
split_1 = (6*2) + 0 + max(6,2)*4
print('Chia sau vị trí 0:',split_0)
print('Chia sau vị trí 1:',split_1)
assert min(split_0,split_1) == 32

Chia sau vị trí 0: 32
Chia sau vị trí 1: 36


## 10. Python: mảng 2 chiều và thứ tự tính theo độ dài

`dp=[[0]*n for _ in range(n)]` tạo n hàng **độc lập**, mỗi hàng n ô. Khác với `[[0]*n]*n` vì phép nhân list có thể làm các hàng cùng trỏ tới một list.

Ta tính các đoạn theo thứ tự **ngắn → dài**. `dp[i][j]` chỉ dùng các đoạn trái/phải **ngắn hơn**, nên khi cần đều đã có kết quả.

In [5]:
n = 3
table = [[0]*n for _ in range(n)]
table[0][1] = 7
print('Bảng 2D:',table)
assert table[1][1] == 0  # Hàng 1 độc lập với hàng 0.
for length in range(2,4):
    print('Độ dài',length,':',[(i,i+length-1) for i in range(n-length+1)])

Bảng 2D: [[0, 7, 0], [0, 0, 0], [0, 0, 0]]
Độ dài 2 : [(0, 1), (1, 2)]
Độ dài 3 : [(0, 2)]


## 11. Lời giải DP `O(n³)` — học trước

Bước 1: tạo bảng `mx[i][j]` trong `O(n²)`.

Bước 2: tạo bảng `dp` bằng 0 trên đường chéo (đoạn 1 lá).

Bước 3: với mỗi độ dài từ 2 tới n, mỗi đầu i, cuối j, thử tất cả k. Dùng công thức ở mục 9.

Có `O(n²)` đoạn và `O(n)` điểm chia mỗi đoạn ⇒ `O(n³)` thời gian, `O(n²)` bộ nhớ. Với `n<=40`, DP vẫn hoàn toàn khả thi.

In [6]:
def mct_interval_dp(arr):
    n = len(arr)
    mx = [[0]*n for _ in range(n)]
    for i in range(n):
        current = 0
        for j in range(i,n):
            current = max(current,arr[j])
            mx[i][j] = current
    dp = [[0]*n for _ in range(n)]
    for length in range(2,n+1):
        for i in range(n-length+1):
            j = i+length-1
            dp[i][j] = min(
                dp[i][k] + dp[k+1][j] + mx[i][k]*mx[k+1][j]
                for k in range(i,j)
            )
    return dp[0][n-1]

assert mct_interval_dp([6,2,4]) == 32
assert mct_interval_dp([4,11]) == 44
assert mct_interval_dp([3,3,3]) == 18
print('Interval DP: PASS')

Interval DP: PASS


## 12. Trace DP — `[6,2,4]`

| Đoạn | Điểm chia k | Chi phí | dp |
|---|---|---:|---:|
| `[6]`, `[2]`, `[4]` | không có | 0 | 0 |
| `[6,2]` | giữa 6 và 2 | `6×2=12` | 12 |
| `[2,4]` | giữa 2 và 4 | `2×4=8` | 8 |
| `[6,2,4]` | k=0 | `0+8+6×4=32` | 32 |
| `[6,2,4]` | k=1 | `12+0+6×4=36` | **min=32** |

Đây là cách nhìn rất quan trọng: **mỗi dp của đoạn lớn được xây từ dp của các đoạn con nhỏ**.

In [7]:
def trace_interval_dp(arr):
    n=len(arr)
    mx=[[0]*n for _ in range(n)]
    for i in range(n):
        for j in range(i,n):
            mx[i][j]=max(arr[i:j+1])
    dp=[[0]*n for _ in range(n)]
    for length in range(2,n+1):
        for i in range(n-length+1):
            j=i+length-1
            candidates=[]
            for k in range(i,j):
                cost=dp[i][k]+dp[k+1][j]+mx[i][k]*mx[k+1][j]
                candidates.append((k,cost))
            dp[i][j]=min(cost for _,cost in candidates)
            print('đoạn',arr[i:j+1],'| điểm chia, chi phí:',candidates,'| dp=',dp[i][j])
    return dp[0][n-1]

assert trace_interval_dp([6,2,4]) == 32

đoạn [6, 2] | điểm chia, chi phí: [(0, 12)] | dp= 12
đoạn [2, 4] | điểm chia, chi phí: [(1, 8)] | dp= 8
đoạn [6, 2, 4] | điểm chia, chi phí: [(0, 32), (1, 36)] | dp= 32


## 13. Bước đột phá: nhận ra cách ghép phần tử nhỏ

DP đã giải đúng. Nhưng mục tiêu là **chọn cách ghép sao cho phần tử nhỏ không bị nhân với phần tử quá lớn khi không cần thiết**.

Cách tham lam trực giác: trong dãy các giá trị lá đang còn, tìm **giá trị nhỏ nhất** `m`; ghép nó với **hàng xóm kề có giá trị nhỏ hơn** trong hai hàng xóm (nếu chỉ có một thì chọn người đó), cộng `m×neighbor`, rồi loại `m` khỏi danh sách.

Quy tắc này có thể được chứng minh bằng phép đổi cấu trúc cây: trong một cây tối ưu luôn tồn tại cách ghép lá nhỏ nhất trước với một hàng xóm có giá trị nhỏ hơn giữa hai bên mà không tăng tổng; sau khi ghép, giá trị lớn hơn vẫn đại diện cho max lá của phần cây mới. Lặp lại quy tắc cho đến khi còn một giá trị.

Lời giải dưới đây là `O(n²)` do phải tìm min/xóa trong list; nó giúp **hình dung** việc mà stack sẽ tối ưu.

In [8]:
def mct_greedy_remove_smallest(arr):
    work = arr[:]
    cost = 0
    while len(work)>1:
        i=min(range(len(work)),key=lambda k: work[k])
        left = work[i-1] if i>0 else float('inf')
        right = work[i+1] if i+1<len(work) else float('inf')
        cost += work[i]*min(left,right)
        work.pop(i)
    return cost

assert mct_greedy_remove_smallest([6,2,4]) == 32
assert mct_greedy_remove_smallest([4,11]) == 44
print('Greedy loại phần tử nhỏ nhất: PASS')

Greedy loại phần tử nhỏ nhất: PASS


## 14. Monotonic Stack là gì?

**Stack (ngăn xếp)**: phần tử thêm vào cuối sẽ được lấy ra trước (`append` và `pop`). `stack[-1]` là đỉnh stack.

**Monotonic decreasing stack**: các giá trị từ đáy lên đỉnh **giảm dần**. Khi có một giá trị mới `x`, nếu đỉnh `mid <= x`, ta bỏ `mid` ra vì đã thấy một giá trị phía phải đủ lớn để “ghép” nó.

Khi pop `mid`, phần tử stack còn lại bên trái là `left`, giá trị mới ở bên phải là `x`. Ta trả giá `mid × min(left,x)`. Chọn **hàng xóm lớn hơn nhỏ hơn**, không dùng `max`.

Đặt `float('inf')` ở đáy stack làm **lính canh** (*sentinel*) để tránh phải xét riêng trường hợp không có phần tử lớn hơn bên trái.

In [9]:
stack_demo=[float('inf'),6,2]
right=4
mid=stack_demo.pop()  # lấy 2
left=stack_demo[-1]   # còn 6
paid=mid*min(left,right)
print('mid=',mid,'left=',left,'right=',right,'giá phải trả=',paid)
stack_demo.append(right)
print('stack sau xử lý:',stack_demo)
assert paid == 8

mid= 2 left= 6 right= 4 giá phải trả= 8
stack sau xử lý: [inf, 6, 4]


## 15. Thuật toán Stack `O(n)` — từ cấu trúc đến code

Với mỗi phần tử `x` trong `arr`:

1. Trong khi `stack[-1] <= x`, pop ra `mid` và cộng `mid * min(stack[-1],x)`.
2. Thêm `x` vào stack.

Cuối cùng, các giá trị chưa ghép còn nằm **giảm dần** trên stack. Pop dần từ phải về trái, cộng `mid * stack[-1]` (không tính lính canh `inf`).

**Tại sao nhanh?** Mỗi giá trị được push **một lần** và pop **tối đa một lần**. Tổng số vòng `while` qua cả chương trình là `O(n)`, dù trông giống vòng lặp lồng nhau.

In [10]:
def mct_monotonic_stack(arr):
    stack = [float('inf')]
    total = 0
    for x in arr:
        while stack[-1] <= x:
            mid = stack.pop()
            total += mid * min(stack[-1],x)
        stack.append(x)
    while len(stack)>2:  # còn sentinel và ít nhất 2 giá trị
        mid = stack.pop()
        total += mid * stack[-1]
    return total

assert mct_monotonic_stack([6,2,4]) == 32
assert mct_monotonic_stack([4,11]) == 44
assert mct_monotonic_stack([1,2,3]) == 8
assert mct_monotonic_stack([3,3,3]) == 18
print('Monotonic stack: PASS')

Monotonic stack: PASS


## 16. Trace bằng tay — Stack trên `[6,2,4]`

| Thao tác | Đỉnh/giá trị | Stack (không tính sentinel) | Cộng chi phí | Tổng |
|---|---|---|---:|---:|
| push 6 | 6 | `[6]` | 0 | 0 |
| push 2 | 2 | `[6,2]` | 0 | 0 |
| x=4, pop 2 | `min(6,4)=4` | `[6]` | `2×4=8` | 8 |
| push 4 | 4 | `[6,4]` | 0 | 8 |
| Dọn stack, pop 4 | trái=6 | `[6]` | `4×6=24` | **32** |

`inf` chỉ làm lính canh. Nó **không được nhân với số nào** trong kết quả cuối.

In [11]:
def trace_monotonic_stack(arr):
    stack=[float('inf')]
    total=0
    for x in arr:
        print('Đọc x =',x,'| stack trước:',stack[1:])
        while stack[-1] <= x:
            mid=stack.pop()
            partner=min(stack[-1],x)
            total += mid*partner
            print('   pop',mid,'ghép với',partner,'cộng',mid*partner,'→ tổng',total)
        stack.append(x)
        print('   push',x,'| stack sau:',stack[1:])
    while len(stack)>2:
        mid=stack.pop()
        partner=stack[-1]
        total+=mid*partner
        print('Dọn stack: pop',mid,'ghép',partner,'→ tổng',total)
    return total

assert trace_monotonic_stack([6,2,4])==32

Đọc x = 6 | stack trước: []
   push 6 | stack sau: [6]
Đọc x = 2 | stack trước: [6]
   push 2 | stack sau: [6, 2]
Đọc x = 4 | stack trước: [6, 2]
   pop 2 ghép với 4 cộng 8 → tổng 8
   push 4 | stack sau: [6, 4]
Dọn stack: pop 4 ghép 6 → tổng 32


## 17. Vì sao tham lam + stack đúng? (lý do thay vì ghi nhớ)

**Ý tưởng đổi cây:** Xét một lá có giá trị nhỏ nhất `m` trong các phần cây còn lại. Khi nó ghép với lá/hợp phần bên cạnh, nút mới có giá `m × b` (với b là max lá của phần đối diện). Giữ thứ tự lá, ta có thể chọn ghép m trước với **b nhỏ hơn trong hai lá/hợp phần kề nhau** mà không làm tăng tổng chi phí; phần còn lại được đại diện bởi giá trị lớn hơn. Lặp quy tắc này xây được một cây tối ưu.

**Vì sao stack mô phỏng được lựa chọn đó?** Khi x tới, tất cả `mid <= x` ở đỉnh stack đã gặp một phần tử phía phải đủ lớn. Phía trái của mid là `stack[-1]` (một giá trị ≥ mid). Chi phí tốt nhất để loại mid khỏi vai trò “max đại diện” là `mid * min(trái, phải)`. Stack giữ các đại diện chưa thể quyết định bằng một dãy giảm. Nếu cuối lượt chưa gặp phần tử bên phải lớn hơn, ta ghép lần lượt các phần tử giảm dần còn lại với bên trái.

**Lưu ý:** đây là lập luận theo tính chất đặc biệt **giá trị nút trong = tích của hai max lá** và **được giữ đúng thứ tự**. Không dùng quy tắc này cho bài DP cây bất kỳ nếu chưa có chứng minh riêng.

## 18. So sánh độ phức tạp

Gọi `n = len(arr)`.

| Cách | Thời gian | Bộ nhớ phụ | Khi nào dùng |
|---|---|---|---|
| Đệ quy thử mọi cách chia không nhớ | **Tăng theo cấp số nhân** (rất chậm) | `O(n)` sâu đệ quy, chưa kể slice | Chỉ ví dụ nhỏ |
| Interval DP | `O(n³)` | `O(n²)` | Dễ suy luận và chứng minh |
| Greedy tìm/xóa phần tử nhỏ nhất trong list | `O(n²)` | `O(n)` | Bước đệm để hiểu stack |
| **Monotonic stack** | **`O(n)`** | **`O(n)`** | Bản nộp đề xuất |

Với n ≤ 40, DP hoàn toàn đủ khả năng chạy; stack cho thấy cách suy ra thuật toán tuyến tính từ cấu trúc bài toán.

## 19. Code nộp LeetCode — một method duy nhất

Code nộp cần `class Solution`, method đúng tên `mctFromLeafValues`, trả `int`. `float('inf')` hoạt động được ở vai trò lính canh vì không bị đưa vào phép cộng kết quả khi dọn stack.

In [12]:
from typing import List

class Solution:
    def mctFromLeafValues(self, arr: List[int]) -> int:
        stack = [float('inf')]
        ans = 0
        for x in arr:
            while stack[-1] <= x:
                mid = stack.pop()
                ans += mid * min(stack[-1], x)
            stack.append(x)
        while len(stack) > 2:
            ans += stack.pop() * stack[-1]
        return ans

assert Solution().mctFromLeafValues([6,2,4])==32
assert Solution().mctFromLeafValues([4,11])==44
print('Code LeetCode: ví dụ đề PASS')

Code LeetCode: ví dụ đề PASS


## 20. Kiểm thử mẫu, biên, số bằng nhau và đơn điệu

Dễ sai khi:

- mảng **2 phần tử**: chỉ có 1 nút trong;
- mảng **tăng dần** hoặc **giảm dần**: stack sẽ pop theo các kiểu khác nhau;
- **giá trị trùng** (ví dụ `[3,3,3]`) cần hiểu `<=` trong điều kiện pop;
- giá trị bằng 1 và bằng 15 (biên constraints).

Các expected dưới đây lấy từ tính tay hoặc DP để tránh phụ thuộc lẫn nhau.

In [13]:
tests = [
    ([6,2,4],32),
    ([4,11],44),
    ([1,1],1),
    ([15,15],225),
    ([3,3,3],18),
    ([1,2,3],8),
    ([3,2,1],8),
    ([1,1,1,1],3),
    ([6,2,4,5],58),
]
for arr, expected in tests:
    assert mct_interval_dp(arr) == expected, (arr,expected,mct_interval_dp(arr))
    assert mct_greedy_remove_smallest(arr) == expected
    assert mct_monotonic_stack(arr) == expected
    assert Solution().mctFromLeafValues(arr) == expected
print('Test mẫu & biên:',len(tests),'PASS')

Test mẫu & biên: 9 PASS


## 21. Kiểm thử đối chiếu ngẫu nhiên — DP là oracle

Sinh các mảng độ dài ngắn để so **4 cách** (vét cạn, DP, xóa nhỏ nhất, stack); và các mảng lớn hơn để so DP/greedy/stack. Dùng random seed cố định và asserts. Nếu một cách sai, chương trình dừng ở đúng mảng phản ví dụ.

In [14]:
import random
rng=random.Random(1130)
for _ in range(110):
    size=rng.randint(2,7)
    arr=[rng.randint(1,15) for _ in range(size)]
    oracle=mct_bruteforce(arr)
    assert mct_interval_dp(arr)==oracle,arr
    assert mct_greedy_remove_smallest(arr)==oracle,arr
    assert mct_monotonic_stack(arr)==oracle,arr
    assert Solution().mctFromLeafValues(arr)==oracle,arr
print('110 ca ngắn, 4 phương pháp: PASS')

for _ in range(70):
    size=rng.randint(8,40)
    arr=[rng.randint(1,15) for _ in range(size)]
    oracle=mct_interval_dp(arr)
    assert mct_greedy_remove_smallest(arr)==oracle,arr
    assert mct_monotonic_stack(arr)==oracle,arr
    assert Solution().mctFromLeafValues(arr)==oracle,arr
print('70 ca tới giới hạn n=40: PASS')

110 ca ngắn, 4 phương pháp: PASS
70 ca tới giới hạn n=40: PASS


## 22. Các lỗi thường gặp (sai → vì sao → sửa)

**Lỗi 1 — nhầm `min` và `max`:** dùng `mid*max(left,right)` làm chi phí cao hơn. Đúng là `mid*min(left,right)`.

**Lỗi 2 — quên dọn stack sau vòng for:** với `[6,2,4]`, mới tính 8 nhưng phải cộng thêm `4×6=24`.

**Lỗi 3 — dừng khi `len(stack)>1`:** stack có sentinel `inf`; không được pop phần tử cuối cùng rồi nhân với `inf`. Phải dùng `while len(stack)>2`.

**Lỗi 4 — cho rằng nút gốc = tổng của hai max:** đề yêu cầu **tích**, không phải tổng.

**Lỗi 5 — tính max của mọi nút con:** đề yêu cầu max của các **lá** trong mỗi cây con; nút trong có giá trị lớn không được dùng làm lá.

In [15]:
def wrong_without_cleanup(arr):
    stack=[float('inf')]
    ans=0
    for x in arr:
        while stack[-1] <= x:
            mid=stack.pop()
            ans+=mid*min(stack[-1],x)
        stack.append(x)
    return ans  # SAI: còn các giá trị chờ trong stack

print('[6,2,4] bỏ quên cleanup:',wrong_without_cleanup([6,2,4]))
assert wrong_without_cleanup([6,2,4]) == 8
assert Solution().mctFromLeafValues([6,2,4]) == 32

[6,2,4] bỏ quên cleanup: 8


## 23. Mẹo nhận diện pattern và chuyển giao

Khi đề có:

- **cấu trúc trên một dãy con liên tiếp** phải giữ nguyên thứ tự ⇒ thử **Interval DP** (`dp[i][j]`).
- Cần lặp lại một quyết định “**đợi gặp số lớn hơn gần nhất**” ⇒ cân nhắc **monotonic stack**.
- Mỗi phần tử chỉ cần ghép với một hàng xóm rồi bị loại ⇒ nghĩ tới **tham lam loại nhỏ nhất** và cách làm nhanh hơn.

Đây là dấu hiệu để **nghĩ thử**, không có nghĩa mọi bài trên cây hay trên đoạn đều dùng được cùng thuật toán stack này.

In [16]:
# Biến thể: chỉ trả chi phí lần ghép đầu (loại số nhỏ nhất) để thấy quy tắc tham lam.
def first_greedy_merge(arr):
    i = min(range(len(arr)), key=lambda k: arr[k])
    left = arr[i-1] if i>0 else float('inf')
    right = arr[i+1] if i+1<len(arr) else float('inf')
    return (i, arr[i]*min(left,right))

assert first_greedy_merge([6,2,4]) == (1,8)
assert first_greedy_merge([4,11]) == (0,44)
print('Biến thể một bước ghép: PASS')

Biến thể một bước ghép: PASS


## 24. Năm bài tập tăng dần — tự làm trước

**Bài 1 — Tính tay:** Với `[6,2,4]`, vẽ hai cách đặt ngoặc, ghi từng giá trị nút trong và tổng.

**Bài 2 — Dự đoán:** Với `[1,2,3]` và `[3,2,1]`, chi phí nhỏ nhất mỗi mảng là bao nhiêu? Giải thích.

**Bài 3 — TODO 2D:** Viết `exercise_interval_max(arr)` trả về bảng `mx[i][j]` của các đoạn i≤j (các ô i>j có thể giữ 0).

**Bài 4 — TODO DP:** Viết `exercise_interval_dp(arr)` bằng công thức với điểm chia k; không gọi hàm lời giải có sẵn.

**Bài 5 — TODO Stack:** Viết `exercise_monotonic_stack(arr)` đầy đủ, có sentinel, `while stack[-1] <= x` và cleanup.

Những function chứa `pass` dưới đây là **chỗ tự viết**; test chỉ bật sau khi hoàn thành.

In [17]:
def exercise_interval_max(arr):
    # TODO: mx[i][j] = max arr[i..j] cho i<=j
    pass

def exercise_interval_dp(arr):
    # TODO: sử dụng bảng mx, xét length tăng từ 2 đến n, thử mọi k
    pass

def exercise_monotonic_stack(arr):
    # TODO: stack=[inf], khi mid<=x thì pop và cộng tích, cuối dọn stack
    pass

# Sau khi làm, tự bật các test:
# assert exercise_interval_max([6,2,4])[0] == [6,6,6]
# assert exercise_interval_dp([6,2,4]) == 32
# assert exercise_monotonic_stack([6,2,4]) == 32

## 25. Gợi ý 3 mức — chưa phải đáp án

| Bài | Gợi ý 1 — trực giác | Gợi ý 2 — cấu trúc | Gợi ý 3 — gần code |
|---|---|---|---|
| 1 | Có đúng 2 cách chia 3 lá | `(6,(2,4))` và `((6,2),4)` | Tích nút trong: 8+24 và 12+24 |
| 2 | Phần tử 2 luôn phải ghép 1 lần | Ghép với hàng xóm nhỏ hơn | `1×2 + 2×3` |
| 3 | Duyệt đầu i, mở rộng cuối j | Dùng biến `current` | `current=max(current,arr[j])` |
| 4 | Một đoạn có thể chia ở nhiều k | `dp[i][i]=0` | `dp[i][k]+dp[k+1][j]+mx[i][k]*mx[k+1][j]` |
| 5 | Pop phần tử nhỏ khi thấy phần tử đủ lớn | `stack[-1] <= x` | `mid*min(stack[-1],x)` sau pop |

## 26. Đáp án tham khảo — đặt sau bài tập

**Bài 1:** cây `(6,(2,4))` cho `8+24=32`; cây `((6,2),4)` cho `12+24=36`. Chọn **32**.

**Bài 2:** cả `[1,2,3]` và `[3,2,1]` đều có đáp án **8**: kết hợp `1×2=2`, sau đó `2×3=6`; tổng 8.

Code tham khảo 3–5 nằm dưới. Mục tiêu là bạn tự viết lại sau khi xem, không học thuộc cú pháp mà không hiểu quá trình.

In [18]:
def answer_interval_max(arr):
    n=len(arr)
    mx=[[0]*n for _ in range(n)]
    for i in range(n):
        running=0
        for j in range(i,n):
            running=max(running,arr[j])
            mx[i][j]=running
    return mx

def answer_interval_dp(arr):
    n=len(arr)
    mx=answer_interval_max(arr)
    dp=[[0]*n for _ in range(n)]
    for length in range(2,n+1):
        for i in range(n-length+1):
            j=i+length-1
            dp[i][j]=min(dp[i][k]+dp[k+1][j]+mx[i][k]*mx[k+1][j]
                         for k in range(i,j))
    return dp[0][n-1]

def answer_monotonic_stack(arr):
    stack=[float('inf')]
    cost=0
    for x in arr:
        while stack[-1] <= x:
            mid=stack.pop()
            cost+=mid*min(stack[-1],x)
        stack.append(x)
    while len(stack)>2:
        cost+=stack.pop()*stack[-1]
    return cost

assert answer_interval_max([6,2,4])[0]==[6,6,6]
assert answer_interval_dp([6,2,4])==32
assert answer_monotonic_stack([6,2,4])==32
print('Đáp án tham khảo 3–5: PASS')

Đáp án tham khảo 3–5: PASS


## 27. Template tư duy + Checklist nộp bài

```text
Giữ thứ tự lá → nghĩ Interval DP với dp[i][j]
Nút trong gốc = max_lá_trái * max_lá_phải
Đoạn một lá có cost 0
Nhận ra quy tắc ghép số nhỏ → nghĩ monotonic stack giảm
Pop mid khi gặp x >= mid; cộng mid * min(left, x)
Cuối vòng for nhớ cleanup stack
```

- [ ] Tôi chỉ cộng **nút trong**, không cộng giá trị lá.
- [ ] Tôi không thay đổi thứ tự các lá.
- [ ] Tôi dùng **nhân** và **max lá** đúng theo đề.
- [ ] Tôi giải thích được `dp[i][j]` và điểm chia k.
- [ ] Tôi tự chạy tay stack trên `[6,2,4]` được 32.
- [ ] Tôi nhớ cleanup stack và không nhân sentinel `inf`.
- [ ] Tôi test mảng hai phần tử, tăng/giảm dần, số trùng, số 1 và 15.